# Huawei-Inspired Healthcare Machine Learning Lab 4
## Decision Tree for Heart-Disease Classification

**Healthcare task:** Use a decision tree to classify records as showing absence or presence of heart disease.

**Official dataset:** UCI Statlog (Heart)

- Dataset page: https://archive.ics.uci.edu/dataset/145/statlog+heart
- Dataset DOI: https://doi.org/10.24432/C57303
- Licence: CC BY 4.0

This notebook adapts the Huawei HCIA-AI V4.0 decision-tree experiment. The Huawei lab converts data into numbers, trains a decision tree using the entropy criterion, visualises the tree, and predicts a new sample. This healthcare version follows the same learning sequence.

**Educational use only:** This notebook is not a diagnostic or clinical decision-support system.

## How to read this notebook

- A line beginning with `#` is a comment written for you. Python ignores it.
- Run the cells from top to bottom.
- A **decision tree** works like a flowchart made of yes/no questions.
- A **feature** is a measurement used to make a decision.
- A **label** is the answer the tree tries to predict.
- The model finds patterns in historical data; it does not apply clinical guidelines.

## Learning objectives

By the end of this lab, learners should be able to:

1. Load a public healthcare dataset.
2. Prepare features and binary labels.
3. Divide data into training and testing sets.
4. Train a decision tree using entropy.
5. Visualise and read a decision tree.
6. Predict a class and probability.
7. Calculate accuracy, sensitivity, specificity, precision, F1 score and ROC-AUC.
8. Explain overfitting and why tree depth may need to be limited.

## Step 1 — Install and import the required packages

In [ ]:
# Install the helper package that downloads datasets from the official UCI repository.
!pip -q install ucimlrepo

# NumPy helps with numerical calculations.
import numpy as np
# Pandas helps us work with tables, similar to an Excel sheet.
import pandas as pd
# Matplotlib helps us draw charts.
import matplotlib.pyplot as plt

# This function downloads the dataset from UCI.
from ucimlrepo import fetch_ucirepo
# This divides the data into training and testing groups.
from sklearn.model_selection import train_test_split
# This is the machine-learning model used in this lab.
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
# These functions calculate classification performance.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

# This fixed number makes the train/test split repeatable.
RANDOM_STATE = 42
print('Packages imported successfully.')

## Step 2 — Download the UCI Statlog Heart dataset

The dataset contains 270 records and 13 features. UCI reports no missing values. Its original target uses:

- `1` = absence of heart disease
- `2` = presence of heart disease

This notebook changes the target to:

- `0` = absence
- `1` = presence


In [ ]:
# Download UCI dataset number 145.
dataset = fetch_ucirepo(id=145)

# X contains the measurements used by the model.
X = dataset.data.features.copy()
# The target table contains the answer for each record.
target_table = dataset.data.targets.copy()

# Select the first target column, regardless of its exact column name.
original_target = pd.to_numeric(target_table.iloc[:, 0], errors='coerce')
# Change the original labels: 1 becomes 0, and 2 becomes 1.
y = original_target.map({1: 0, 2: 1})

# Convert every feature into a number.
X = X.apply(pd.to_numeric, errors='coerce')

# Combine X and y temporarily so incomplete rows can be removed safely.
complete_data = X.copy()
complete_data['target'] = y
complete_data = complete_data.dropna()

# Separate the cleaned table back into features and target.
X = complete_data.drop(columns='target')
y = complete_data['target'].astype(int)

print('Dataset name:', dataset.metadata.name)
print('Number of usable records:', X.shape[0])
print('Number of features:', X.shape[1])
print('Feature names:', list(X.columns))
X.head()

## Step 3 — Understand the coded features

Some categories are already written as numbers in this dataset. Examples include:

- `sex`: a binary code
- `chest-pain`: four category codes
- `fasting-blood-sugar`: binary code
- `electrocardiographic`: three category codes
- `angina`: binary code
- `thal`: category codes

The numbers represent categories; they are not scores of severity.

In [ ]:
# Show the UCI variable-information table.
# This helps students see which variables are continuous, binary or categorical.
dataset.variables

## Step 4 — Examine the two outcome groups

In [ ]:
# Count the records in each class.
class_counts = y.value_counts().sort_index()

# Build a readable summary table.
class_table = pd.DataFrame({
    'Class label': [0, 1],
    'Meaning': ['Absence', 'Presence'],
    'Number of records': [class_counts.get(0, 0), class_counts.get(1, 0)]
})

class_table

In [ ]:
# Draw a bar chart of the outcome groups.
plt.figure(figsize=(6, 4))
plt.bar(class_table['Meaning'], class_table['Number of records'])
plt.ylabel('Number of records')
plt.title('Heart-Disease Class Distribution')
plt.show()

## Step 5 — Divide the data into training and testing sets

The tree learns from the training set. The test set is kept separate to evaluate how the model handles records it has not seen before.

In [ ]:
# Use 80% of the data for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    # Stratify preserves a similar class balance in both groups.
    stratify=y
)

print('Training records:', len(X_train))
print('Testing records:', len(X_test))
print('Presence proportion in training data:', round(y_train.mean(), 3))
print('Presence proportion in testing data:', round(y_test.mean(), 3))

## Step 6 — Build the decision tree

The Huawei lab uses the `entropy` criterion. Entropy helps the tree choose questions that separate the classes.

We limit the tree to a depth of 3 so that it remains readable and is less likely to memorise the training data.

In [ ]:
# Create the decision-tree model.
decision_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=3,
    min_samples_leaf=10,
    random_state=RANDOM_STATE
)

# Ask the tree to learn rules from the training data.
decision_tree.fit(X_train, y_train)
print('Decision tree training completed.')

## Step 7 — Visualise the tree

How to read each box:

- The first line is the question used for splitting.
- `entropy` shows how mixed the classes are at that point.
- `samples` is the number of training records in that box.
- `value` shows how many records belong to each class.
- `class` is the majority prediction in that box.

In [ ]:
# Draw the tree directly inside the notebook.
plt.figure(figsize=(24, 12))
plot_tree(
    decision_tree,
    feature_names=list(X.columns),
    class_names=['Absence', 'Presence'],
    filled=True,
    rounded=True,
    proportion=False,
    precision=2,
    fontsize=9
)
plt.title('Decision Tree for Heart-Disease Classification')
plt.tight_layout()
plt.show()

## Step 8 — Save the decision-tree diagram as a PDF

In [ ]:
# Create the same tree diagram again for saving.
plt.figure(figsize=(24, 12))
plot_tree(
    decision_tree,
    feature_names=list(X.columns),
    class_names=['Absence', 'Presence'],
    filled=True,
    rounded=True,
    precision=2,
    fontsize=9
)
plt.title('Decision Tree for Heart-Disease Classification')
plt.tight_layout()

# Save the diagram in the Colab files area.
tree_pdf_name = 'heart_disease_decision_tree.pdf'
plt.savefig(tree_pdf_name, format='pdf', bbox_inches='tight')
plt.close()

print('Saved:', tree_pdf_name)

## Step 9 — Display the tree as written rules

The same tree can be shown as text. Indentation indicates movement from the top of the tree towards its branches.

In [ ]:
# Convert the tree into simple written rules.
tree_rules = export_text(
    decision_tree,
    feature_names=list(X.columns),
    decimals=2
)

print(tree_rules)

## Step 10 — Predict the test records

The tree produces a class and a probability for every test record.

In [ ]:
# Predict absence or presence for each test record.
y_pred = decision_tree.predict(X_test)
# Obtain the predicted probability of heart-disease presence.
y_probability = decision_tree.predict_proba(X_test)[:, 1]

# Show the first ten predictions beside the correct answers.
prediction_preview = pd.DataFrame({
    'Actual label': y_test.to_numpy()[:10],
    'Predicted label': y_pred[:10],
    'Predicted presence probability': np.round(y_probability[:10], 3)
})

prediction_preview

## Step 11 — Calculate the confusion matrix

With presence defined as the positive class:

- **True positive:** presence correctly predicted as presence
- **True negative:** absence correctly predicted as absence
- **False positive:** absence incorrectly predicted as presence
- **False negative:** presence incorrectly predicted as absence

In [ ]:
# Calculate the four confusion-matrix values.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
true_negative, false_positive, false_negative, true_positive = cm.ravel()

print('True negatives:', true_negative)
print('False positives:', false_positive)
print('False negatives:', false_negative)
print('True positives:', true_positive)

In [ ]:
# Draw the confusion matrix.
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Absence', 'Presence']
)
display.plot(values_format='d')
plt.title('Confusion Matrix')
plt.show()

## Step 12 — Calculate performance measures

- **Accuracy:** proportion of all predictions that are correct
- **Sensitivity:** proportion of presence records correctly detected
- **Specificity:** proportion of absence records correctly identified
- **Precision:** among records predicted as presence, the proportion that truly show presence
- **F1 score:** balance between precision and sensitivity
- **ROC-AUC:** ability of the probabilities to separate the two classes across many thresholds

In [ ]:
# Calculate the main classification metrics.
accuracy = accuracy_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred, pos_label=1)
specificity = true_negative / (true_negative + false_positive)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

# Place the results in a readable table.
metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1 score', 'ROC-AUC'],
    'Value': [accuracy, sensitivity, specificity, precision, f1, roc_auc]
})
metrics_table['Value'] = metrics_table['Value'].round(3)
metrics_table

## Step 13 — Display the detailed classification report

In [ ]:
# Show precision, recall and F1 score for each class.
print(classification_report(
    y_test,
    y_pred,
    target_names=['Absence', 'Presence'],
    digits=3,
    zero_division=0
))

## Step 14 — Draw the ROC curve

In [ ]:
# Draw the ROC curve from the predicted probabilities.
RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title('ROC Curve')
plt.show()

## Step 15 — Examine feature importance

Feature importance shows which variables the tree used most often and most effectively for its splits. It does not prove that a feature causes heart disease.

In [ ]:
# Create a table of feature-importance values.
importance_table = pd.DataFrame({
    'Feature': X.columns,
    'Importance': decision_tree.feature_importances_
}).sort_values('Importance', ascending=False)

importance_table

In [ ]:
# Draw only features that the tree actually used.
used_features = importance_table[importance_table['Importance'] > 0]

plt.figure(figsize=(9, 5))
plt.barh(used_features['Feature'][::-1], used_features['Importance'][::-1])
plt.xlabel('Feature importance')
plt.title('Features Used by the Decision Tree')
plt.tight_layout()
plt.show()

## Step 16 — Predict one fictional example

The notebook starts with the median training record and changes several fields. This ensures that every required feature is present. The example is fictional and is not intended to represent a real patient.

In [ ]:
# Start with the median value of every training feature.
fictional_record = X_train.median().to_frame().T

# Replace selected values with fictional values.
# The loop checks that a column exists before changing it.
fictional_values = {
    'age': 58,
    'sex': 1,
    'chest-pain': 4,
    'rest-bp': 140,
    'serum-chol': 250,
    'fasting-blood-sugar': 0,
    'electrocardiographic': 0,
    'max-heart-rate': 135,
    'angina': 1,
    'oldpeak': 2.0
}

for column_name, value in fictional_values.items():
    if column_name in fictional_record.columns:
        fictional_record.loc[fictional_record.index[0], column_name] = value

# Predict the class and the probability of presence.
fictional_class = decision_tree.predict(fictional_record)[0]
fictional_probability = decision_tree.predict_proba(fictional_record)[0, 1]
class_name = 'Presence' if fictional_class == 1 else 'Absence'

print('Predicted class:', class_name)
print(f'Predicted presence probability: {fictional_probability:.3f}')
fictional_record

## Step 17 — Demonstrate overfitting

A very deep tree may memorise the training data. It can achieve excellent training accuracy but perform less well on unseen test data.

In [ ]:
# Build a tree without a depth limit.
unrestricted_tree = DecisionTreeClassifier(
    criterion='entropy',
    random_state=RANDOM_STATE
)
unrestricted_tree.fit(X_train, y_train)

# Compare training and testing accuracy for both trees.
overfitting_table = pd.DataFrame({
    'Model': ['Readable limited tree', 'Unrestricted tree'],
    'Training accuracy': [
        accuracy_score(y_train, decision_tree.predict(X_train)),
        accuracy_score(y_train, unrestricted_tree.predict(X_train))
    ],
    'Testing accuracy': [
        accuracy_score(y_test, decision_tree.predict(X_test)),
        accuracy_score(y_test, unrestricted_tree.predict(X_test))
    ],
    'Tree depth': [decision_tree.get_depth(), unrestricted_tree.get_depth()],
    'Number of leaves': [decision_tree.get_n_leaves(), unrestricted_tree.get_n_leaves()]
})

overfitting_table.round(3)

## Student exercises

1. Change `max_depth` from 3 to 2, 4 and 5.
2. Change `min_samples_leaf` from 10 to 5 and 20.
3. Replace `criterion='entropy'` with `criterion='gini'`.
4. Compare training accuracy with testing accuracy.
5. Count the false negatives and explain why they matter.
6. Identify the first question at the root of the tree.
7. Explain why a readable tree is not automatically a clinically valid rule.

## Responsible-use notes

- The dataset is intended for teaching and research.
- The output is a statistical classification, not a diagnosis.
- Do not enter identifiable patient data into this notebook.
- The tree may reflect limitations or biases in the historical dataset.
- A decision tree can appear understandable while still being inaccurate or clinically inappropriate.
- Clinical use requires external validation, fairness assessment, governance, clinician oversight and regulatory review.

## Dataset citation

*Statlog (Heart)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C57303